# Chapter 9 — Backtracking

*Built with [Claude](https://claude.ai) by Anthropic.*

Backtracking is DFS on a decision tree — build a candidate incrementally, prune branches that can't lead to a valid solution, and undo choices when you backtrack. It's exhaustive search made efficient through early stopping.

In [1]:
from typing import List
from itertools import combinations, permutations as iperms

def show_decision(curr, choice, action):
    """Print one step of the decision tree."""
    print(f"  {'  ' * len(curr)}{action} {choice!r}  → curr={curr}")

print('  Helpers loaded')
print('  itertools.combinations and permutations available for DS approach')

  Helpers loaded
  itertools.combinations and permutations available for DS approach


---

## Part 1 — Subsets / Power Set

**DS/ML connection:** Exhaustive feature subset enumeration — `itertools.combinations` is the DS shortcut; backtracking is how you implement it, and understanding it lets you add constraints (e.g., minimum subset size, budget constraints) that `itertools` can't express.

In [2]:
# ── DS/ML PARALLEL: itertools combinations ──
import itertools

nums = [1, 2, 3]
# Power set via itertools
power_set = []
for r in range(len(nums) + 1):
    power_set.extend(list(c) for c in itertools.combinations(nums, r))
print(f"  itertools power set of {nums}:")
print(f"  {power_set}")

  itertools power set of [1, 2, 3]:
  [[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]]


### LC 78 — [Subsets](https://leetcode.com/problems/subsets/)

**Problem:** Return all possible subsets (the power set) of a list of unique integers.

**DS parallel:** Enumerate all feature subsets for exhaustive feature selection.

**Key insight:** Append `curr[:]` at every recursion level (not just leaves). Use a `start` index to avoid revisiting earlier elements.

In [3]:
# ── DS/ML APPROACH: itertools ──
def subsets_itertools(nums):
    result = []
    for r in range(len(nums) + 1):
        result.extend(list(c) for c in itertools.combinations(nums, r))
    return result

# ── RAW PYTHON (interview answer) ──
def subsets(nums: List[int]) -> List[List[int]]:
    result = []
    def backtrack(start, curr):
        result.append(curr[:])   # snapshot at every level
        for i in range(start, len(nums)):
            curr.append(nums[i])
            backtrack(i + 1, curr)
            curr.pop()           # UNCHOOSE
    backtrack(0, [])
    return result

nums = [1, 2, 3]
r1 = sorted(subsets_itertools(nums))
r2 = sorted(subsets(nums))
print(f"  nums={nums}")
print(f"  itertools: {r1}")
print(f"  backtrack: {r2}")
print(f"  match: {r1 == r2}")

  nums=[1, 2, 3]
  itertools: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
  backtrack: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
  match: True


In [4]:
# ── TRACE: LC 78 ──
def subsets_trace(nums):
    result = []
    def backtrack(start, curr, depth=0):
        snapshot = curr[:]
        result.append(snapshot)
        indent = '  ' * depth
        print(f"  {indent}record {snapshot}")
        for i in range(start, len(nums)):
            print(f"  {indent}choose {nums[i]}")
            curr.append(nums[i])
            backtrack(i + 1, curr, depth + 1)
            curr.pop()
            print(f"  {indent}unchoose {nums[i]}")
    backtrack(0, [])
    print(f"  all subsets: {result}")

subsets_trace([1, 2, 3])

  record []
  choose 1
    record [1]
    choose 2
      record [1, 2]
      choose 3
        record [1, 2, 3]
      unchoose 3
    unchoose 2
    choose 3
      record [1, 3]
    unchoose 3
  unchoose 1
  choose 2
    record [2]
    choose 3
      record [2, 3]
    unchoose 3
  unchoose 2
  choose 3
    record [3]
  unchoose 3
  all subsets: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


### LC 90 — [Subsets II](https://leetcode.com/problems/subsets-ii/) (with duplicates)

**Problem:** Same as LC 78 but input may contain duplicates. Return unique subsets only.

**DS parallel:** Enumerate unique feature subsets when the feature list has redundant (duplicate) columns.

**Key insight:** Sort first, then skip `if i > start and nums[i] == nums[i-1]` to avoid generating duplicate subsets at the same recursion level.

In [5]:
# ── DS/ML APPROACH: itertools then dedupe ──
def subsetsWithDup_itertools(nums):
    result = set()
    for r in range(len(nums) + 1):
        for c in itertools.combinations(sorted(nums), r):
            result.add(c)
    return sorted([list(s) for s in result])

# ── RAW PYTHON (interview answer) ──
def subsetsWithDup(nums: List[int]) -> List[List[int]]:
    nums.sort()
    result = []
    def backtrack(start, curr):
        result.append(curr[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:   # skip duplicates
                continue
            curr.append(nums[i])
            backtrack(i + 1, curr)
            curr.pop()
    backtrack(0, [])
    return result

nums = [1, 2, 2]
r1 = subsetsWithDup_itertools(nums)
r2 = sorted(subsetsWithDup(nums[:]))
print(f"  nums={nums}")
print(f"  itertools (dedupe): {r1}")
print(f"  backtrack (prune):  {r2}")

  nums=[1, 2, 2]
  itertools (dedupe): [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
  backtrack (prune):  [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


---

## Part 2 — Permutations and Combinations with Constraints

**DS/ML connection:** Trying all orderings of pipeline stages, searching hyperparameter combinations that hit a target — the constrained versions add pruning that `itertools` doesn't support natively.

### LC 46 — [Permutations](https://leetcode.com/problems/permutations/)

**Problem:** Return all permutations of a list of distinct integers.

**DS parallel:** `itertools.permutations` — trying all orderings of processing stages in a pipeline.

**Key insight:** Track which elements are `used`. At each position, try every unused element, mark it used, recurse, then unmark.

In [6]:
# ── DS/ML APPROACH: itertools ──
def permute_itertools(nums):
    return [list(p) for p in itertools.permutations(nums)]

# ── RAW PYTHON (interview answer) ──
def permute(nums: List[int]) -> List[List[int]]:
    result = []
    def backtrack(curr, remaining):
        if not remaining:
            result.append(curr[:])
            return
        for i in range(len(remaining)):
            curr.append(remaining[i])
            backtrack(curr, remaining[:i] + remaining[i+1:])
            curr.pop()
    backtrack([], nums)
    return result

nums = [1, 2, 3]
r1 = sorted(permute_itertools(nums))
r2 = sorted(permute(nums))
print(f"  nums={nums}")
print(f"  itertools: {r1}")
print(f"  backtrack: {r2}")
print(f"  count: {len(r2)} (= {len(nums)}! = {__import__('math').factorial(len(nums))})")

  nums=[1, 2, 3]
  itertools: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
  backtrack: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
  count: 6 (= 3! = 6)


In [7]:
# ── TRACE: LC 46 ──
def permute_trace(nums):
    result = []
    def backtrack(curr, remaining, depth=0):
        indent = '  ' * depth
        if not remaining:
            result.append(curr[:])
            print(f"  {indent}✓ record {curr}")
            return
        for i in range(len(remaining)):
            choice = remaining[i]
            print(f"  {indent}choose {choice}  remaining={remaining[:i]+remaining[i+1:]}")
            curr.append(choice)
            backtrack(curr, remaining[:i] + remaining[i+1:], depth + 1)
            curr.pop()
    backtrack([], nums)
    print(f"  all permutations: {result}")

permute_trace([1, 2, 3])

  choose 1  remaining=[2, 3]
    choose 2  remaining=[3]
      choose 3  remaining=[]
        ✓ record [1, 2, 3]
    choose 3  remaining=[2]
      choose 2  remaining=[]
        ✓ record [1, 3, 2]
  choose 2  remaining=[1, 3]
    choose 1  remaining=[3]
      choose 3  remaining=[]
        ✓ record [2, 1, 3]
    choose 3  remaining=[1]
      choose 1  remaining=[]
        ✓ record [2, 3, 1]
  choose 3  remaining=[1, 2]
    choose 1  remaining=[2]
      choose 2  remaining=[]
        ✓ record [3, 1, 2]
    choose 2  remaining=[1]
      choose 1  remaining=[]
        ✓ record [3, 2, 1]
  all permutations: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### LC 39 — [Combination Sum](https://leetcode.com/problems/combination-sum/)

**Problem:** Given `candidates` (distinct, positive) and a `target`, return all combinations that sum to `target`. Candidates can be reused.

**DS parallel:** Find all hyperparameter combinations (with repetition allowed) that achieve a target score.

**Key insight:** Sort candidates so you can break early when `candidates[i] > remaining`. Recur from the same index `i` (not `i+1`) to allow reuse.

In [8]:
# ── DS/ML APPROACH: brute force via itertools with_replacement ──
from itertools import combinations_with_replacement

def combinationSum_brute(candidates, target):
    result = []
    max_count = target // min(candidates) + 1
    for r in range(1, max_count + 1):
        for combo in combinations_with_replacement(candidates, r):
            if sum(combo) == target:
                result.append(sorted(combo))
    return sorted(result)

# ── RAW PYTHON (interview answer) ──
def combinationSum(candidates: List[int], target: int) -> List[List[int]]:
    candidates.sort()
    result = []
    def backtrack(start, curr, remaining):
        if remaining == 0:
            result.append(curr[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:   # pruning
                break
            curr.append(candidates[i])
            backtrack(i, curr, remaining - candidates[i])  # i, not i+1 (allow reuse)
            curr.pop()
    backtrack(0, [], target)
    return result

candidates = [2, 3, 6, 7]
target = 7
r1 = combinationSum_brute(candidates, target)
r2 = sorted(combinationSum(candidates[:], target))
print(f"  candidates={candidates}, target={target}")
print(f"  brute:     {r1}")
print(f"  backtrack: {r2}")

  candidates=[2, 3, 6, 7], target=7
  brute:     [[2, 2, 3], [7]]
  backtrack: [[2, 2, 3], [7]]


In [9]:
# ── TRACE: LC 39 ──
def combinationSum_trace(candidates, target):
    candidates = sorted(candidates)
    result = []
    def backtrack(start, curr, remaining, depth=0):
        indent = '  ' * depth
        if remaining == 0:
            result.append(curr[:])
            print(f"  {indent}✓ found {curr}")
            return
        for i in range(start, len(candidates)):
            c = candidates[i]
            if c > remaining:
                print(f"  {indent}{c} > {remaining}: prune")
                break
            print(f"  {indent}try {c}  remaining={remaining-c}")
            curr.append(c)
            backtrack(i, curr, remaining - c, depth + 1)
            curr.pop()
    backtrack(0, [], target)
    print(f"  all combinations: {result}")

combinationSum_trace([2, 3, 6, 7], 7)

  try 2  remaining=5
    try 2  remaining=3
      try 2  remaining=1
        2 > 1: prune
      try 3  remaining=0
        ✓ found [2, 2, 3]
      6 > 3: prune
    try 3  remaining=2
      3 > 2: prune
    6 > 5: prune
  try 3  remaining=4
    try 3  remaining=1
      3 > 1: prune
    6 > 4: prune
  try 6  remaining=1
    6 > 1: prune
  try 7  remaining=0
    ✓ found [7]
  all combinations: [[2, 2, 3], [7]]
